# 🧬 Pipeline A: One-Hot Encoding → mCNN (Baseline)

## Architecture
```
DNA Sequence (101bp string)
       │
       ▼
┌──────────────────────────┐
│   One-Hot Encoding       │
│   A=[1,0,0,0]            │
│   C=[0,1,0,0]            │
│   G=[0,0,1,0]            │
│   T=[0,0,0,1]            │
│   → (batch, 101, 4)      │
└──────────────────────────┘
       │
       ▼
┌──────────────────────────┐
│   Multi-Scale CNN        │  embedding_dim=4
│   Conv1D k=[3,5,7,9]    │
│   → GlobalMaxPool        │
│   → FC → 4 classes       │
└──────────────────────────┘
       │
       ▼
  SP1 / SP2 / SP4 / Negative
```

### Purpose
Establish the **baseline** accuracy when mCNN only sees raw nucleotide identity.
No pre-trained language model, no structural features — just A/C/G/T.

⚡ **No GPU required** — runs in ~30s on CPU.

### Cell 1: Setup

In [ ]:
!pip install -q scikit-learn matplotlib seaborn

import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

if os.path.basename(os.getcwd()) == REPO_NAME:
    os.chdir("..")

if not os.path.isdir(REPO_NAME):
    print("Cloning...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"Ready: {os.getcwd()}")

### Cell 2: Load Data + One-Hot Encode

In [ ]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from src.mcnn_model import MultiScaleCNN
from src.train import train_model, evaluate_model, plot_curves

# --- Load FASTA ---
def load_fasta(path):
    seqs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

# --- One-Hot Encoding (keeps 2D shape for mCNN) ---
def seqs_to_onehot_2d(sequences):
    """Convert DNA sequences to (N, 101, 4) one-hot tensor."""
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    n = len(sequences)
    seq_len = len(sequences[0])
    onehot = np.zeros((n, seq_len, 4), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j, nuc in enumerate(seq):
            if nuc in mapping:
                onehot[i, j, mapping[nuc]] = 1.0
    return onehot

print("Loading datasets...")
seqs_sp1 = load_fasta("data/processed/sp1_positive_final.fasta")
seqs_sp2 = load_fasta("data/processed/sp2_positive_final.fasta")
seqs_sp4 = load_fasta("data/processed/sp4_positive_final.fasta")
seqs_neg = load_fasta("data/processed/negative_final.fasta")

sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
y = np.concatenate([
    np.zeros(len(seqs_sp1)),
    np.ones(len(seqs_sp2)),
    np.full(len(seqs_sp4), 2),
    np.full(len(seqs_neg), 3)
])

print(f"Total: {len(sequences)} sequences")
print(f"Distribution: {dict(zip(['SP1','SP2','SP4','Neg'], np.bincount(y.astype(int))))}")

# --- One-Hot encode ---
print("\nOne-Hot encoding...")
X = seqs_to_onehot_2d(sequences)
print(f"Feature tensor shape: {X.shape}  (samples, positions, channels)")

### Cell 3: Build mCNN + DataLoaders

In [ ]:
# --- Stratified Split ---
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}")

# --- TensorDatasets ---
train_ds = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train).long()
)
val_ds = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val).long()
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")

# --- mCNN with embedding_dim=4 (one-hot channels) ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = MultiScaleCNN(
    embedding_dim=4,        # ← 4 channels (A/C/G/T) instead of 768
    branch_channels=128,
    num_classes=4,
    dropout_rate=0.5
)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")

### Cell 4: Train

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=30,              # More epochs since one-hot is weaker signal
    lr=0.001,
    device=device,
    output_dir='models'
)

### Cell 5: Evaluate + Plots

In [ ]:
from IPython.display import Image, display

# Training curves
plot_curves(history, save_dir='figures')
display(Image('figures/mcnn_training_curves.png'))

# Load best checkpoint and evaluate
best_path = os.path.join('models', 'best_mcnn_model.pt')
model.load_state_dict(torch.load(best_path, map_location=device))
print(f"Loaded best model from {best_path}")

class_names = ['SP1', 'SP2', 'SP4', 'Negative']
evaluate_model(model, val_loader, class_names, device=device, save_dir='figures')

print("\n--- Confusion Matrix ---")
display(Image('figures/confusion_matrix.png'))
print("\n--- ROC Curves ---")
display(Image('figures/roc_curves.png'))
print("\n--- Precision-Recall Curves ---")
display(Image('figures/precision_recall_curves.png'))